# Station maps for Czechia

Builds four figures: the whole country and an eastern crop, each with stations
coloured by **elevation** and by **mean snow season length**.

The terrain comes from SRTM via Google Earth Engine, so nothing large is
downloaded and the local 3 km DEM is not needed here. Run the cells in order.

In [ ]:
# Rasterio reads the GeoTIFF that Earth Engine returns, including its
# georeferencing, so the stations land in the right place on the terrain.
# Colab does not ship with it. Takes about 20 seconds.
!pip install -q rasterio
print("done")

In [ ]:
import io, json, zipfile
from pathlib import Path

import ee
import numpy as np
import pandas as pd
import requests
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, Normalize
from matplotlib.lines import Line2D

# Your Earth Engine project, already registered for noncommercial academic use.
EE_PROJECT = "ee-aleynaturan886"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

## The station table

Get `map_stations.csv` into Colab, then run the cell below.

**The easy way:** click the **folder icon** in the left sidebar, then the
**upload button** at the top of that panel (a page with an up-arrow), and choose
`results/tables/05_maps/map_stations.csv` from your repository. Wait for the
progress circle beside the filename to finish.

Drag-and-drop into that panel works too, but the upload button is more reliable.

The cell looks in several places and tells you exactly what to do if it cannot
find the file, so run it either way.

In [ ]:
from pathlib import Path

CANDIDATES = [
    "map_stations.csv",
    "/content/map_stations.csv",
    "/content/drive/MyDrive/map_stations.csv",
]

path = next((p for p in CANDIDATES if Path(p).exists()), None)

if path is None:
    here = sorted(p.name for p in Path("/content").glob("*") if p.is_file())
    raise SystemExit(
        "map_stations.csv is not in Colab yet.\n\n"
        "Click the folder icon in the left sidebar, then the upload button at the\n"
        "top of that panel, and choose it from\n"
        "  results/tables/05_maps/map_stations.csv\n"
        "Wait for the upload circle to finish, then run this cell again.\n\n"
        f"Files currently in /content: {here if here else 'none'}"
    )

st = pd.read_csv(path)

needed = ["station_id", "station_name", "latitude", "longitude",
          "elevation_m", "selected", "mean_snow_duration_d", "notable"]
missing = [c for c in needed if c not in st.columns]
if missing:
    raise SystemExit(f"These columns are missing: {missing}\n"
                     f"The file has: {list(st.columns)}")

print(f"read from   : {path}")
print(f"{len(st)} stations, {int(st['selected'].sum())} selected")
print(f"longitude   : {st.longitude.min():.2f} to {st.longitude.max():.2f}")
print(f"latitude    : {st.latitude.min():.2f} to {st.latitude.max():.2f}")
print(f"elevation   : {st.elevation_m.min():.0f} to {st.elevation_m.max():.0f} m")
print(f"labelled    : {int((st.notable.fillna('') != '').sum())} stations")
st.head(3)

## Settings

Everything adjustable lives here. The extents are generous padding around the
station coordinates; the eastern box covers the Beskydy and Jeseníky, where the
high stations are.

In [ ]:
# Map extents as [lon_min, lat_min, lon_max, lat_max]
EXTENT_CZ   = [12.00, 48.50, 19.00, 51.10]
EXTENT_EAST = [16.30, 49.20, 18.95, 50.60]

# Terrain resolution in metres. Smaller means sharper and a bigger download.
# 250 m over the whole country is about 3000 x 1150 pixels.
SCALE_CZ, SCALE_EAST = 250, 100

# "terrain" colours the relief by height, which is what your supervisor asked
# for. "grey" makes the background neutral so the station colours carry all the
# information. Try both and see which reads better.
BACKGROUND = "terrain"

# Colour limits for the station markers, from the ranges your script reported.
ELEV_LIMS = (150, 1350)      # metres
DUR_LIMS  = (20, 225)        # days

# Station colour schemes. Both are ColorBrewer sequential ramps that
# ColorBrewer itself flags as colourblind safe, so the figures stay readable
# for deuteranopia and protanopia, which affect roughly 8 percent of men.
# Other safe ColorBrewer options, if you want to try them:
#   YlOrBr  YlOrRd  OrRd  YlGn  YlGnBu  GnBu  BuPu  PuBu  PuBuGn  Blues  Greens
CMAP_ELEV = "YlOrBr"         # warm, so the dots separate from the terrain
CMAP_DUR  = "YlGnBu"         # cool, longer season reads as deeper blue

DEM_SOURCE = "SRTM"          # or "COPERNICUS" for GLO-30, slower but cleaner

In [ ]:
def get_dem(extent, scale):
    """Download a terrain tile from Earth Engine as a numpy array.

    Returns the elevation grid plus the extent matplotlib needs for imshow.
    The GeoTIFF carries its own georeferencing, so the bounds come from the
    file rather than from the box we asked for, which avoids half-pixel drift.
    """
    region = ee.Geometry.Rectangle(extent, "EPSG:4326", False)

    if DEM_SOURCE == "COPERNICUS":
        img = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM").mosaic()
    else:
        img = ee.Image("USGS/SRTMGL1_003").select("elevation")

    approx_px = ((extent[2] - extent[0]) * 111320 / scale) * \
                ((extent[3] - extent[1]) * 111320 / scale)
    print(f"  requesting about {approx_px/1e6:.1f} million pixels at {scale} m")

    url = img.getDownloadURL({"region": region, "scale": scale,
                              "format": "GEO_TIFF", "crs": "EPSG:4326"})
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    print(f"  downloaded {len(r.content)/1e6:.1f} MB")

    with rasterio.open(io.BytesIO(r.content)) as src:
        dem = src.read(1).astype("float32")
        b = src.bounds
        tr = src.transform
    dem[dem < -1000] = np.nan            # sea and no-data

    # Pixel size in METRES, needed for sensible hillshading. The grid is in
    # degrees, and a degree of longitude is shorter than a degree of latitude
    # at this latitude, so the two differ.
    lat_mid = (b.bottom + b.top) / 2
    dx = abs(tr.a) * 111320 * np.cos(np.radians(lat_mid))
    dy = abs(tr.e) * 111320
    return dem, [b.left, b.right, b.bottom, b.top], dx, dy


def get_border():
    """Czech national boundary, simplified, as a list of lon/lat rings."""
    fc = (ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
            .filter(ee.Filter.eq("country_na", "Czechia")))
    geom = fc.geometry().simplify(500).getInfo()
    polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    return [np.array(ring) for poly in polys for ring in poly]


print("Whole country:")
dem_cz, ext_cz, dx_cz, dy_cz = get_dem(EXTENT_CZ, SCALE_CZ)
print("Eastern crop:")
dem_e, ext_e, dx_e, dy_e = get_dem(EXTENT_EAST, SCALE_EAST)
border = get_border()
print(f"\ncountry grid {dem_cz.shape}, east grid {dem_e.shape}, "
      f"{len(border)} border rings")

## The plotting function

One function draws every version. `colour_by` picks which station property the
colour scale shows.

In [ ]:
def scale_bar(ax, extent, km, y_frac=0.05, x_frac=0.06):
    """Simple scale bar. One degree of longitude is about 71.5 km at 50 N."""
    km_per_deg = 111.32 * np.cos(np.radians((extent[2] + extent[3]) / 2))
    dlon = km / km_per_deg
    x0 = extent[0] + x_frac * (extent[1] - extent[0])
    y0 = extent[2] + y_frac * (extent[3] - extent[2])
    h = 0.012 * (extent[3] - extent[2])
    for i, frac in enumerate([(0, .5), (.5, 1)]):
        ax.add_patch(plt.Rectangle((x0 + frac[0]*dlon, y0), (frac[1]-frac[0])*dlon, h,
                                   facecolor="k" if i == 0 else "w",
                                   edgecolor="k", lw=.6, zorder=6))
    for v, lab in [(0, "0"), (dlon/2, f"{km//2:.0f}"), (dlon, f"{km:.0f} km")]:
        ax.text(x0 + v, y0 + h*1.5, lab, ha="center", va="bottom",
                fontsize=7.5, zorder=6)


def north_arrow(ax, extent, x_frac=0.945, y_frac=0.86):
    x = extent[0] + x_frac * (extent[1] - extent[0])
    y = extent[2] + y_frac * (extent[3] - extent[2])
    d = 0.055 * (extent[3] - extent[2])
    ax.annotate("", xy=(x, y + d), xytext=(x, y),
                arrowprops=dict(arrowstyle="-|>", color="k", lw=1.4), zorder=6)
    ax.text(x, y + d*1.12, "N", ha="center", va="bottom",
            fontsize=10, fontweight="bold", zorder=6)


def draw_map(dem, extent, dx, dy, colour_by, title, inset_box=None,
             label_stations=True, figsize=(11, 8)):
    """extent is [lon_min, lon_max, lat_min, lat_max] as imshow wants it."""
    cfg = {
        "elevation_m":          dict(cmap=CMAP_ELEV, lims=ELEV_LIMS,
                                     label="Station elevation (m a.s.l.)"),
        "mean_snow_duration_d": dict(cmap=CMAP_DUR,  lims=DUR_LIMS,
                                     label="Mean snow season length (days)"),
    }[colour_by]

    fig, ax = plt.subplots(figsize=figsize)

    # --- terrain -----------------------------------------------------------
    ls = LightSource(azdeg=315, altdeg=45)
    filled = np.nan_to_num(dem, nan=np.nanmin(dem))
    if BACKGROUND == "terrain":
        rgb = ls.shade(filled, cmap=plt.cm.terrain, blend_mode="soft",
                       vert_exag=2.5, dx=dx, dy=dy)
        rgb = 1 - (1 - rgb) * 0.55          # fade it so stations stay readable
    else:
        hs = ls.hillshade(filled, vert_exag=2.5, dx=dx, dy=dy)
        rgb = np.dstack([0.55 + 0.45*hs]*3)
    ax.imshow(rgb, extent=extent, origin="upper", interpolation="bilinear", zorder=1)

    # --- national boundary -------------------------------------------------
    for ring in border:
        ax.plot(ring[:, 0], ring[:, 1], color="#222222", lw=1.1, zorder=3)

    # --- stations ----------------------------------------------------------
    inbox = st[(st.longitude.between(extent[0], extent[1])) &
               (st.latitude.between(extent[2], extent[3]))]
    other = inbox[~inbox.selected]
    used = inbox[inbox.selected].dropna(subset=[colour_by])

    ax.scatter(other.longitude, other.latitude, s=9, c="#9e9e9e",
               edgecolor="none", alpha=.55, zorder=4)
    sc = ax.scatter(used.longitude, used.latitude, s=46, c=used[colour_by],
                    cmap=cfg["cmap"], norm=Normalize(*cfg["lims"]),
                    edgecolor="k", linewidth=.45, zorder=5)

    # --- labels for the notable stations -----------------------------------
    # Several of the notable stations sit in the same massif, so a fixed
    # offset would stack the boxes on top of each other. Stations that fall
    # close together are grouped, and the group's labels are fanned out
    # vertically with leader lines pointing back to each dot.
    if label_stations and "notable" in used.columns:
        lab = used[used.notable.fillna("") != ""].copy()
        lab = lab.sort_values(["latitude", "longitude"], ascending=[False, True])
        span_x = extent[1] - extent[0]
        span_y = extent[3] - extent[2]
        near_x, near_y = span_x * 0.07, span_y * 0.12

        # group the ones that sit in the same massif
        groups = []
        for _, r in lab.iterrows():
            for grp in groups:
                if (abs(r.longitude - grp[0].longitude) < near_x
                        and abs(r.latitude - grp[0].latitude) < near_y):
                    grp.append(r); break
            else:
                groups.append([r])

        gap = max(span_y * 0.050, 0.055)    # vertical gap between stacked boxes
        pad = span_x * 0.035                # how far the stack sits from the dots

        for grp in groups:
            lons = [r.longitude for r in grp]
            lats = [r.latitude for r in grp]
            # put the stack on whichever side of the map has room
            right = (sum(lons) / len(lons)) < (extent[0] + extent[1]) / 2
            tx = (max(lons) + pad) if right else (min(lons) - pad)
            ha = "left" if right else "right"
            # all the boxes of one group share an x and are evenly spaced in y,
            # so they cannot overlap each other whatever the dots do
            grp = sorted(grp, key=lambda r: -r.latitude)
            mid = sum(lats) / len(lats)
            n = len(grp)
            for k, r in enumerate(grp):
                ty = mid - gap * (k - (n - 1) / 2)
                ty = min(max(ty, extent[2] + gap), extent[3] - gap)
                ax.annotate(r.station_name, xy=(r.longitude, r.latitude),
                            xytext=(tx, ty), textcoords="data",
                            fontsize=8.5, ha=ha, va="center", zorder=7,
                            bbox=dict(boxstyle="round,pad=0.25", fc="white",
                                      ec="#999999", lw=.5, alpha=.92),
                            arrowprops=dict(arrowstyle="-", lw=.7,
                                            color="#444444",
                                            shrinkA=0, shrinkB=3))

    # --- the eastern box drawn on the country map --------------------------
    if inset_box:
        ax.add_patch(plt.Rectangle((inset_box[0], inset_box[1]),
                                   inset_box[2]-inset_box[0], inset_box[3]-inset_box[1],
                                   fill=False, ec="#c0392b", lw=1.6, ls="--", zorder=6))
        ax.text(inset_box[0], inset_box[3], " eastern crop", color="#c0392b",
                fontsize=8.5, va="top", ha="left", zorder=6,
                bbox=dict(boxstyle="square,pad=0.15", fc="white",
                          ec="none", alpha=.75))

    # --- furniture ---------------------------------------------------------
    ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
    # One degree of latitude is longer on the ground than one of longitude at
    # 50 N, so without this the country comes out squashed.
    ax.set_aspect(1 / np.cos(np.radians((extent[2] + extent[3]) / 2)))
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    ax.set_title(title, fontsize=13, pad=10)
    ax.grid(alpha=.18, lw=.4, ls=":")

    span = extent[1] - extent[0]
    scale_bar(ax, extent, 100 if span > 4 else 50)
    north_arrow(ax, extent)

    cb = fig.colorbar(sc, ax=ax, fraction=.034, pad=.02)
    cb.set_label(cfg["label"], fontsize=9.5)

    ax.legend(handles=[
        Line2D([], [], marker="o", ls="", mfc="#cccccc", mec="k", mew=.45,
               ms=8, label=f"Used in the analysis (n = {len(used)})"),
        Line2D([], [], marker="o", ls="", mfc="#9e9e9e", mec="none",
               ms=5, alpha=.55, label=f"Not selected (n = {len(other)})"),
    ], loc="lower right", fontsize=8.5, framealpha=.92)

    fig.tight_layout()
    return fig

## Make the figures

In [ ]:
figs = {}

figs["map_czechia_elevation.png"] = draw_map(
    dem_cz, ext_cz, dx_cz, dy_cz, "elevation_m",
    "CHMI snow stations and terrain, Czechia",
    inset_box=EXTENT_EAST)

figs["map_czechia_duration.png"] = draw_map(
    dem_cz, ext_cz, dx_cz, dy_cz, "mean_snow_duration_d",
    "Mean snow season length at CHMI stations, 1976 to 2025",
    inset_box=EXTENT_EAST)

figs["map_east_elevation.png"] = draw_map(
    dem_e, ext_e, dx_e, dy_e, "elevation_m",
    "Snow stations and terrain, eastern Czechia", figsize=(10, 8))

figs["map_east_duration.png"] = draw_map(
    dem_e, ext_e, dx_e, dy_e, "mean_snow_duration_d",
    "Mean snow season length, eastern Czechia", figsize=(10, 8))

plt.show()

## Save and download

Writes the four figures at 200 dpi and downloads them as one zip.

In [ ]:
out = Path("maps"); out.mkdir(exist_ok=True)
for name, fig in figs.items():
    fig.savefig(out / name, dpi=200, bbox_inches="tight")
    print(f"saved {name}")

zp = "czechia_station_maps.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out.glob("*.png")):
        z.write(p, p.name)

files.download(zp)